# 🔬 Planilha Embriologia Reconciliation: Local DuckDB Silver vs. AWS Athena (Staging / Prod)

This notebook performs an exhaustive, table-by-table reconciliation between the consolidated local Silver tables (`silver.planilha_embriologia_*`) and the AWS Athena database (`silver_embriologia_staging` / `silver_embriologia_prod`).

### Staging & Production Sheet Nomenclature:
The tables in AWS Athena follow the updated, standardized model naming:
1. **Core Procedure Streams**: `fresh`, `fet`, `recep`, `fot`
2. **Specialized Preservation & Insemination Streams**: `egg_freezing` (Local: `silver.planilha_embriologia_fp_ovulos`), `iui` (Local: `silver.planilha_embriologia_iiu`)
3. **Dedicated Clinical Streams**: `doadoras` (Oocyte Donation) and `fp_semen` (Sperm Cryopreservation)

### Shifting Between Environments:
To shift validation between **Staging** and **Production**, simply change `ATHENA_ENV = 'staging'` or `'prod'` in the configuration cell below.

### Dimensions Analyzed:
* **Row Counts & Volume Variance** (Total & Cohort Breakdown by Year 2021–2026)
* **Master Patient Index Links** (Strategy L Prontuário Matching against Clinisys EMR)
* **Patient Population Overlap** (Distinct PINs: Common, Local-Only, Athena-Only)
* **Clinical Outcome Distributions** (`result`, `opu`, `mii_total`, `mii_crio`, `qtd_blasto`, `no_et`, `no_nascidos`, `gravidez_clinica`, `gravidez_bioquimica`, etc.)


In [ ]:
import os
import re
import warnings
import duckdb
import pandas as pd
import numpy as np
from pyathena import connect
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# ==============================================================================
# ⚙️ ENVIRONMENT CONFIGURATION: Easily shift between STAGING and PROD
# ==============================================================================
# Set ATHENA_ENV to 'staging' or 'prod' to target the respective Athena database:
#   - 'staging' -> silver_embriologia_staging
#   - 'prod'    -> silver_embriologia_prod
# ==============================================================================
ATHENA_ENV = 'staging'  # Options: 'staging' | 'prod'
ATHENA_DB = f'silver_embriologia_{ATHENA_ENV}'

ATHENA_REGION = 'sa-east-1'
ATHENA_WORKGROUP = 'datalake-admins'

# Local DuckDB Database Path
DUCKDB_PATH = '../../database/huntington_data_lake.duckdb'
if not os.path.exists(DUCKDB_PATH):
    DUCKDB_PATH = 'database/huntington_data_lake.duckdb'

# ==============================================================================
# 📋 CLINICAL PROCEDURE STREAM CONFIGURATION
# Standardized mapping between Stream Name, Local DuckDB Silver Table, and Athena Table
# ==============================================================================
STREAM_CONFIGS = [
    ('FET',       'planilha_embriologia_fet',       'fet'),
    ('FRESH',     'planilha_embriologia_fresh',     'fresh'),
    ('RECEP',     'planilha_embriologia_recep',     'recep'),
    ('FOT',       'planilha_embriologia_fot',       'fot'),
    ('FP_OVULOS', 'planilha_embriologia_fp_ovulos', 'egg_freezing'),
    ('IIU',       'planilha_embriologia_iiu',       'iui'),
    ('DOADORAS',  'planilha_embriologia_doadoras',  'doadoras'),
    ('FP_SEMEN',  'planilha_embriologia_fp_semen',  'fp_semen'),
]

print(f"📁 Local DuckDB:      {DUCKDB_PATH}")
print(f"☁️  AWS Athena Env:    {ATHENA_ENV.upper()} -> Catalog: awsdatacatalog, Database: {ATHENA_DB} (Region: {ATHENA_REGION}, Workgroup: {ATHENA_WORKGROUP})")


In [ ]:
# Verify database connectivity
try:
    with duckdb.connect(DUCKDB_PATH, read_only=True) as d_conn:
        silver_tables = [r[0] for r in d_conn.execute("SELECT table_name FROM information_schema.tables WHERE table_schema='silver' AND table_name LIKE 'planilha_embriologia_%'").fetchall()]
    print(f"✅ Local DuckDB Connected: Found {len(silver_tables)} Planilha Embriologia tables in schema 'silver':")
    for t in sorted(silver_tables):
        print(f"   • silver.{t}")
except Exception as e:
    print(f"❌ Local DuckDB Connection Failed: {e}")

try:
    with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP) as a_conn:
        with a_conn.cursor() as cur:
            cur.execute(f"SHOW TABLES IN {ATHENA_DB}")
            all_ath = [r[0] for r in cur.fetchall()]
            stream_targets = [cfg[2] for cfg in STREAM_CONFIGS]
            ath_tables = [t for t in all_ath if t in stream_targets]
            other_tables = [t for t in all_ath if t not in stream_targets]
    print(f"\n✅ AWS Athena Connected ({ATHENA_ENV.upper()}): Found {len(ath_tables)} active stream tables in '{ATHENA_DB}':")
    for t in sorted(ath_tables):
        print(f"   • {ATHENA_DB}.{t}")
    if other_tables:
        print(f"   (Additional catalog tables in {ATHENA_DB}: {', '.join(sorted(other_tables))})")
except Exception as e:
    print(f"❌ AWS Athena Connection Failed: {e}")


In [ ]:
def run_duck(sql):
    """Execute SQL against Local DuckDB and return pandas DataFrame"""
    with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
        return conn.execute(sql).df()

def run_athena(sql):
    """Execute SQL against AWS Athena and return pandas DataFrame"""
    with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP) as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
            cols = [desc[0] for desc in cur.description] if cur.description else []
            rows = cur.fetchall()
            return pd.DataFrame(rows, columns=cols)


## 📊 Part 1: Global Executive Reconciliation Dashboard
Comparative summary across all 8 clinical procedure streams:


In [ ]:
dashboard_rows = []

for name, loc_t, ath_t in STREAM_CONFIGS:
    loc_df = run_duck(f'''
        SELECT 
            COUNT(*) as total_rows,
            COUNT(DISTINCT pin) as distinct_pins
        FROM silver.{loc_t}
    ''')
    loc_total = loc_df['total_rows'].iloc[0]
    loc_pins = loc_df['distinct_pins'].iloc[0]

    ath_df = run_athena(f'''
        SELECT 
            COUNT(*) as total_rows,
            COUNT(DISTINCT pin) as distinct_pins
        FROM {ATHENA_DB}.{ath_t}
    ''')
    ath_total = ath_df['total_rows'].iloc[0]
    ath_pins = ath_df['distinct_pins'].iloc[0]
    delta = loc_total - ath_total
    
    if delta == 0:
        status = 'Exact 100.00% Parity'
    elif abs(delta) == 1:
        status = f'100% Patient Match ({delta:+d} row)'
    else:
        status = f'Delta: {delta:+d}'

    dashboard_rows.append({
        'Stream': name,
        'Local Silver Rows': f"{loc_total:,}",
        f'Athena {ATHENA_ENV.capitalize()} Rows': f"{ath_total:,}",
        'Row Delta': f"{delta:+,}",
        'Local Distinct PINs': f"{loc_pins:,}",
        f'Athena {ATHENA_ENV.capitalize()} PINs': f"{ath_pins:,}",
        'Parity Status': status
    })

df_dashboard = pd.DataFrame(dashboard_rows)
display(df_dashboard)


## 🔬 Part 2: FRESH (FIV) Table Deep Dive
Comparing `silver.planilha_embriologia_fresh` with Athena `{ATHENA_DB}.fresh`.


In [ ]:
print("=== FRESH: YEARLY BREAKDOWN COMPARISON ===")
fresh_loc_years = run_duck('''
    SELECT 
        COALESCE(REGEXP_EXTRACT(file_name, '202[1-6]'), 'Unknown') as year,
        COUNT(*) as local_rows
    FROM silver.planilha_embriologia_fresh
    GROUP BY 1 ORDER BY 1
''')

fresh_ath_years = run_athena(f'''
    SELECT 
        CAST(year AS VARCHAR) as year,
        COUNT(*) as athena_rows
    FROM {ATHENA_DB}.fresh
    GROUP BY 1 ORDER BY 1
''')

fresh_comp_years = pd.merge(fresh_loc_years, fresh_ath_years, on='year', how='outer').fillna(0)
fresh_comp_years['local_rows'] = fresh_comp_years['local_rows'].astype(int)
fresh_comp_years['athena_rows'] = fresh_comp_years['athena_rows'].astype(int)
fresh_comp_years['delta'] = fresh_comp_years['local_rows'] - fresh_comp_years['athena_rows']
display(fresh_comp_years)

print("\n=== FRESH: FILE-BY-FILE RECONCILIATION ===")
fresh_loc_files = run_duck("SELECT file_name, COUNT(*) as local_rows FROM silver.planilha_embriologia_fresh GROUP BY 1")
fresh_loc_files['file_base'] = fresh_loc_files['file_name'].apply(lambda x: re.split(r'[/\\]', str(x))[-1])
fresh_loc_files = fresh_loc_files.groupby('file_base')['local_rows'].sum().reset_index()

fresh_ath_files = run_athena(f"SELECT file_name, COUNT(*) as athena_rows FROM {ATHENA_DB}.fresh GROUP BY 1")
fresh_ath_files['file_base'] = fresh_ath_files['file_name'].apply(lambda x: re.split(r'[/\\]', str(x))[-1])
fresh_ath_files = fresh_ath_files.groupby('file_base')['athena_rows'].sum().reset_index()

fresh_file_comp = pd.merge(fresh_loc_files, fresh_ath_files, on='file_base', how='outer').fillna(0)
fresh_file_comp['local_rows'] = fresh_file_comp['local_rows'].astype(int)
fresh_file_comp['athena_rows'] = fresh_file_comp['athena_rows'].astype(int)
fresh_file_comp['delta'] = fresh_file_comp['local_rows'] - fresh_file_comp['athena_rows']
fresh_diffs = fresh_file_comp[fresh_file_comp['delta'] != 0].sort_values(by='delta', ascending=False)
if len(fresh_diffs) > 0:
    print(f"Files with differences ({len(fresh_diffs)}):")
    display(fresh_diffs)
else:
    print("All files have 100% exact identical row counts!")

print("\n=== FRESH: CLINICAL & LABORATORY OUTCOME AGGREGATIONS ===")
fresh_loc_outcomes = run_duck('''
    SELECT 
        'Local Silver DuckDB' as source,
        COUNT(*) as total_cycles,
        SUM(TRY_CAST(opu AS BIGINT)) as total_opu_aspirated,
        SUM(TRY_CAST(total_de_mii AS BIGINT)) as total_mature_mii,
        SUM(TRY_CAST(qtd_blasto AS BIGINT)) as total_blastocysts,
        COUNT(CASE WHEN gravidez_clinica IS NOT NULL AND TRIM(gravidez_clinica) NOT IN ('', '0', '0.0') THEN 1 END) as clin_preg,
        COUNT(CASE WHEN gravidez_bioquimica IS NOT NULL AND TRIM(gravidez_bioquimica) NOT IN ('', '0', '0.0') THEN 1 END) as bio_preg,
        SUM(TRY_CAST(no_nascidos AS BIGINT)) as live_births,
        SUM(TRY_CAST(no_et AS BIGINT)) as embryos_transferred
    FROM silver.planilha_embriologia_fresh
''')

fresh_ath_outcomes = run_athena(f'''
    SELECT 
        'Athena {ATHENA_ENV.capitalize()}' as source,
        COUNT(*) as total_cycles,
        SUM(TRY_CAST(opu AS BIGINT)) as total_opu_aspirated,
        SUM(TRY_CAST(total_de_mii AS BIGINT)) as total_mature_mii,
        SUM(TRY_CAST(qtd_blasto AS BIGINT)) as total_blastocysts,
        COUNT(CASE WHEN gravidez_clinica IS NOT NULL AND TRIM(gravidez_clinica) NOT IN ('', '0', '0.0') THEN 1 END) as clin_preg,
        COUNT(CASE WHEN gravidez_bioquimica IS NOT NULL AND TRIM(gravidez_bioquimica) NOT IN ('', '0', '0.0') THEN 1 END) as bio_preg,
        SUM(TRY_CAST(no_nascidos AS BIGINT)) as live_births,
        SUM(TRY_CAST(no_et AS BIGINT)) as embryos_transferred
    FROM {ATHENA_DB}.fresh
''')

display(pd.concat([fresh_loc_outcomes, fresh_ath_outcomes], ignore_index=True))


## ❄️ Part 3: FET (TEC) Table Deep Dive
Comparing `silver.planilha_embriologia_fet` with Athena `{ATHENA_DB}.fet`.


In [ ]:
print("=== FET: YEARLY BREAKDOWN COMPARISON ===")
fet_loc_years = run_duck('''
    SELECT 
        COALESCE(REGEXP_EXTRACT(file_name, '202[1-6]'), 'Unknown') as year,
        COUNT(*) as local_rows
    FROM silver.planilha_embriologia_fet
    GROUP BY 1 ORDER BY 1
''')

fet_ath_years = run_athena(f'''
    SELECT 
        CAST(year AS VARCHAR) as year,
        COUNT(*) as athena_rows
    FROM {ATHENA_DB}.fet
    GROUP BY 1 ORDER BY 1
''')

fet_comp_years = pd.merge(fet_loc_years, fet_ath_years, on='year', how='outer').fillna(0)
fet_comp_years['local_rows'] = fet_comp_years['local_rows'].astype(int)
fet_comp_years['athena_rows'] = fet_comp_years['athena_rows'].astype(int)
fet_comp_years['delta'] = fet_comp_years['local_rows'] - fet_comp_years['athena_rows']
display(fet_comp_years)

print("\n=== FET: FILE-BY-FILE RECONCILIATION ===")
fet_loc_files = run_duck("SELECT file_name, COUNT(*) as local_rows FROM silver.planilha_embriologia_fet GROUP BY 1")
fet_loc_files['file_base'] = fet_loc_files['file_name'].apply(lambda x: re.split(r'[/\\]', str(x))[-1])
fet_loc_files = fet_loc_files.groupby('file_base')['local_rows'].sum().reset_index()

fet_ath_files = run_athena(f"SELECT file_name, COUNT(*) as athena_rows FROM {ATHENA_DB}.fet GROUP BY 1")
fet_ath_files['file_base'] = fet_ath_files['file_name'].apply(lambda x: re.split(r'[/\\]', str(x))[-1])
fet_ath_files = fet_ath_files.groupby('file_base')['athena_rows'].sum().reset_index()

fet_file_comp = pd.merge(fet_loc_files, fet_ath_files, on='file_base', how='outer').fillna(0)
fet_file_comp['local_rows'] = fet_file_comp['local_rows'].astype(int)
fet_file_comp['athena_rows'] = fet_file_comp['athena_rows'].astype(int)
fet_file_comp['delta'] = fet_file_comp['local_rows'] - fet_file_comp['athena_rows']
fet_diffs = fet_file_comp[fet_file_comp['delta'] != 0].sort_values(by='delta', ascending=False)
if len(fet_diffs) > 0:
    print(f"Files with name differences (e.g. IBI vs IBIRA):")
    display(fet_diffs)
else:
    print("All files have 100% exact identical row counts!")

print("\n=== FET: CLINICAL OUTCOME & EMBRYO TRANSFER AGGREGATIONS ===")
fet_outcomes = run_duck('''
    SELECT 
        'Local Silver DuckDB' as source,
        COUNT(*) as total_transfers,
        COUNT(CASE WHEN result IS NOT NULL AND TRIM(result) != '' THEN 1 END) as with_result,
        COUNT(CASE WHEN gravidez_clinica IS NOT NULL AND TRIM(gravidez_clinica) NOT IN ('', '0', '0.0') THEN 1 END) as clin_preg,
        COUNT(CASE WHEN gravidez_bioquimica IS NOT NULL AND TRIM(gravidez_bioquimica) NOT IN ('', '0', '0.0') THEN 1 END) as bio_preg,
        SUM(TRY_CAST(no_et AS BIGINT)) as sum_embryos_transferred,
        SUM(TRY_CAST(no_nascidos AS BIGINT)) as sum_live_births
    FROM silver.planilha_embriologia_fet
''')

fet_ath_outcomes = run_athena(f'''
    SELECT 
        'Athena {ATHENA_ENV.capitalize()}' as source,
        COUNT(*) as total_transfers,
        COUNT(CASE WHEN result IS NOT NULL AND TRIM(result) != '' THEN 1 END) as with_result,
        COUNT(CASE WHEN gravidez_clinica IS NOT NULL AND TRIM(gravidez_clinica) NOT IN ('', '0', '0.0') THEN 1 END) as clin_preg,
        COUNT(CASE WHEN gravidez_bioquimica IS NOT NULL AND TRIM(gravidez_bioquimica) NOT IN ('', '0', '0.0') THEN 1 END) as bio_preg,
        SUM(TRY_CAST(no_et AS BIGINT)) as sum_embryos_transferred,
        SUM(TRY_CAST(no_nascidos AS BIGINT)) as sum_live_births
    FROM {ATHENA_DB}.fet
''')

display(pd.concat([fet_outcomes, fet_ath_outcomes], ignore_index=True))

print("\n=== FET: RESULT VALUE DISTRIBUTION ===")
fet_res_loc = run_duck("SELECT result, COUNT(*) as local_cnt FROM silver.planilha_embriologia_fet GROUP BY 1 ORDER BY 2 DESC LIMIT 6")
fet_res_ath = run_athena(f"SELECT result, COUNT(*) as athena_cnt FROM {ATHENA_DB}.fet GROUP BY 1 ORDER BY 2 DESC LIMIT 6")
display(pd.merge(fet_res_loc, fet_res_ath, on='result', how='outer').fillna(0))


## 🥚 Part 4: RECEP & FOT Tables Deep Dive
Comparing `silver.planilha_embriologia_recep` and `silver.planilha_embriologia_fot` with Athena (`{ATHENA_DB}.recep`, `{ATHENA_DB}.fot`).


In [ ]:
print("=== RECEP: YEARLY BREAKDOWN COMPARISON ===")
recep_loc_years = run_duck('''
    SELECT 
        COALESCE(REGEXP_EXTRACT(file_name, '202[1-6]'), 'Unknown') as year,
        COUNT(*) as local_rows
    FROM silver.planilha_embriologia_recep
    GROUP BY 1 ORDER BY 1
''')

recep_ath_years = run_athena(f'''
    SELECT 
        CAST(year AS VARCHAR) as year,
        COUNT(*) as athena_rows
    FROM {ATHENA_DB}.recep
    GROUP BY 1 ORDER BY 1
''')

recep_comp = pd.merge(recep_loc_years, recep_ath_years, on='year', how='outer').fillna(0)
recep_comp['delta'] = recep_comp['local_rows'].astype(int) - recep_comp['athena_rows'].astype(int)
display(recep_comp)

print("\n=== RECEP: CLINICAL OUTCOMES COMPARISON ===")
rec_loc_outcomes = run_duck('''
    SELECT 
        'Local Silver DuckDB' as source,
        COUNT(*) as total_rows,
        SUM(TRY_CAST(no_et AS BIGINT)) as total_embryos_transferred,
        COUNT(CASE WHEN gravidez_clinica IS NOT NULL AND TRIM(gravidez_clinica) NOT IN ('', '0', '0.0') THEN 1 END) as clin_preg,
        COUNT(CASE WHEN gravidez_bioquimica IS NOT NULL AND TRIM(gravidez_bioquimica) NOT IN ('', '0', '0.0') THEN 1 END) as bio_preg,
        SUM(TRY_CAST(no_nascidos AS BIGINT)) as live_births
    FROM silver.planilha_embriologia_recep
''')
rec_ath_outcomes = run_athena(f'''
    SELECT 
        'Athena {ATHENA_ENV.capitalize()}' as source,
        COUNT(*) as total_rows,
        SUM(TRY_CAST(no_et AS BIGINT)) as total_embryos_transferred,
        COUNT(CASE WHEN gravidez_clinica IS NOT NULL AND TRIM(gravidez_clinica) NOT IN ('', '0', '0.0') THEN 1 END) as clin_preg,
        COUNT(CASE WHEN gravidez_bioquimica IS NOT NULL AND TRIM(gravidez_bioquimica) NOT IN ('', '0', '0.0') THEN 1 END) as bio_preg,
        SUM(TRY_CAST(no_nascidos AS BIGINT)) as live_births
    FROM {ATHENA_DB}.recep
''')
display(pd.concat([rec_loc_outcomes, rec_ath_outcomes], ignore_index=True))

print("\n=== FOT: YEARLY BREAKDOWN COMPARISON ===")
fot_loc_years = run_duck('''
    SELECT 
        COALESCE(REGEXP_EXTRACT(file_name, '202[1-6]'), 'Unknown') as year,
        COUNT(*) as local_rows
    FROM silver.planilha_embriologia_fot
    GROUP BY 1 ORDER BY 1
''')

fot_ath_years = run_athena(f'''
    SELECT 
        CAST(year AS VARCHAR) as year,
        COUNT(*) as athena_rows
    FROM {ATHENA_DB}.fot
    GROUP BY 1 ORDER BY 1
''')

fot_comp = pd.merge(fot_loc_years, fot_ath_years, on='year', how='outer').fillna(0)
fot_comp['delta'] = fot_comp['local_rows'].astype(int) - fot_comp['athena_rows'].astype(int)
display(fot_comp)

print("\n=== FOT: CLINICAL OUTCOMES COMPARISON ===")
fot_loc_outcomes = run_duck('''
    SELECT 
        'Local Silver DuckDB' as source,
        COUNT(*) as total_rows,
        COUNT(CASE WHEN gravidez_clinica IS NOT NULL AND TRIM(gravidez_clinica) NOT IN ('', '0', '0.0') THEN 1 END) as clin_preg,
        COUNT(CASE WHEN gravidez_bioquimica IS NOT NULL AND TRIM(gravidez_bioquimica) NOT IN ('', '0', '0.0') THEN 1 END) as bio_preg,
        SUM(TRY_CAST(no_nascidos AS BIGINT)) as live_births
    FROM silver.planilha_embriologia_fot
''')
fot_ath_outcomes = run_athena(f'''
    SELECT 
        'Athena {ATHENA_ENV.capitalize()}' as source,
        COUNT(*) as total_rows,
        COUNT(CASE WHEN gravidez_clinica IS NOT NULL AND TRIM(gravidez_clinica) NOT IN ('', '0', '0.0') THEN 1 END) as clin_preg,
        COUNT(CASE WHEN gravidez_bioquimica IS NOT NULL AND TRIM(gravidez_bioquimica) NOT IN ('', '0', '0.0') THEN 1 END) as bio_preg,
        SUM(TRY_CAST(no_nascidos AS BIGINT)) as live_births
    FROM {ATHENA_DB}.fot
''')
display(pd.concat([fot_loc_outcomes, fot_ath_outcomes], ignore_index=True))


## 🔬 Part 5: Fertility Preservation (Óvulos) vs. Athena Egg Freezing
Comparing `silver.planilha_embriologia_fp_ovulos` with Athena `{ATHENA_DB}.egg_freezing`.


In [ ]:
print("=== YEARLY INGESTION BREAKDOWN: LOCAL FP_OVULOS vs ATHENA EGG_FREEZING ===")
ef_loc_years = run_duck('''
    SELECT 
        COALESCE(REGEXP_EXTRACT(file_name, '202[1-6]'), 'Unknown') as year,
        COUNT(*) as local_fp_rows
    FROM silver.planilha_embriologia_fp_ovulos
    GROUP BY 1 ORDER BY 1
''')

ef_ath_years = run_athena(f'''
    SELECT 
        CAST(year AS VARCHAR) as year,
        COUNT(*) as athena_rows
    FROM {ATHENA_DB}.egg_freezing
    GROUP BY 1 ORDER BY 1
''')

ef_comp = pd.merge(ef_loc_years, ef_ath_years, on='year', how='outer').fillna(0)
ef_comp['local_fp_rows'] = ef_comp['local_fp_rows'].astype(int)
ef_comp['athena_rows'] = ef_comp['athena_rows'].astype(int)
ef_comp['delta'] = ef_comp['local_fp_rows'] - ef_comp['athena_rows']
display(ef_comp)

print("\n=== FP_OVULOS: FILE-BY-FILE RECONCILIATION ===")
ef_loc_files = run_duck("SELECT file_name, COUNT(*) as local_rows FROM silver.planilha_embriologia_fp_ovulos GROUP BY 1")
ef_loc_files['file_base'] = ef_loc_files['file_name'].apply(lambda x: re.split(r'[/\\]', str(x))[-1])
ef_loc_files = ef_loc_files.groupby('file_base')['local_rows'].sum().reset_index()

ef_ath_files = run_athena(f"SELECT file_name, COUNT(*) as athena_rows FROM {ATHENA_DB}.egg_freezing GROUP BY 1")
ef_ath_files['file_base'] = ef_ath_files['file_name'].apply(lambda x: re.split(r'[/\\]', str(x))[-1])
ef_ath_files = ef_ath_files.groupby('file_base')['athena_rows'].sum().reset_index()

ef_file_comp = pd.merge(ef_loc_files, ef_ath_files, on='file_base', how='outer').fillna(0)
ef_file_comp['local_rows'] = ef_file_comp['local_rows'].astype(int)
ef_file_comp['athena_rows'] = ef_file_comp['athena_rows'].astype(int)
ef_file_comp['delta'] = ef_file_comp['local_rows'] - ef_file_comp['athena_rows']
ef_diffs = ef_file_comp[ef_file_comp['delta'] != 0].sort_values(by='delta', ascending=False)
if len(ef_diffs) > 0:
    print(f"Files with differences ({len(ef_diffs)}):")
    display(ef_diffs)
else:
    print("All files have 100% exact identical row counts!")

print("\n=== CLINICAL PRESERVATION METRICS (LOCAL vs ATHENA) ===")
fp_loc_outcomes = run_duck('''
    SELECT 
        'Local Silver DuckDB' as source,
        COUNT(*) as total_cycles,
        SUM(TRY_CAST(opu AS BIGINT)) as total_oocytes_aspirated,
        SUM(TRY_CAST(mii_crio AS BIGINT)) as total_mii_cryopreserved
    FROM silver.planilha_embriologia_fp_ovulos
''')
fp_ath_outcomes = run_athena(f'''
    SELECT 
        'Athena {ATHENA_ENV.capitalize()}' as source,
        COUNT(*) as total_cycles,
        SUM(TRY_CAST(opu AS BIGINT)) as total_oocytes_aspirated,
        SUM(TRY_CAST(mii_crio AS BIGINT)) as total_mii_cryopreserved
    FROM {ATHENA_DB}.egg_freezing
''')
display(pd.concat([fp_loc_outcomes, fp_ath_outcomes], ignore_index=True))


## 🔬 Part 6: IIU (Inseminação Intrauterina) vs. Athena IUI
Comparing `silver.planilha_embriologia_iiu` with Athena `{ATHENA_DB}.iui`.


In [ ]:
print("=== YEARLY INGESTION BREAKDOWN: LOCAL IIU vs ATHENA IUI ===")
iiu_loc_years = run_duck('''
    SELECT 
        COALESCE(REGEXP_EXTRACT(file_name, '202[1-6]'), 'Unknown') as year,
        COUNT(*) as local_rows
    FROM silver.planilha_embriologia_iiu
    GROUP BY 1 ORDER BY 1
''')

iiu_ath_years = run_athena(f'''
    SELECT 
        CAST(year AS VARCHAR) as year,
        COUNT(*) as athena_rows
    FROM {ATHENA_DB}.iui
    GROUP BY 1 ORDER BY 1
''')

iiu_comp = pd.merge(iiu_loc_years, iiu_ath_years, on='year', how='outer').fillna(0)
iiu_comp['local_rows'] = iiu_comp['local_rows'].astype(int)
iiu_comp['athena_rows'] = iiu_comp['athena_rows'].astype(int)
iiu_comp['delta'] = iiu_comp['local_rows'] - iiu_comp['athena_rows']
display(iiu_comp)

print("\n=== IIU: FILE-BY-FILE RECONCILIATION ===")
iiu_loc_files = run_duck("SELECT file_name, COUNT(*) as local_rows FROM silver.planilha_embriologia_iiu GROUP BY 1")
iiu_loc_files['file_base'] = iiu_loc_files['file_name'].apply(lambda x: re.split(r'[/\\]', str(x))[-1])
iiu_loc_files = iiu_loc_files.groupby('file_base')['local_rows'].sum().reset_index()

iiu_ath_files = run_athena(f"SELECT file_name, COUNT(*) as athena_rows FROM {ATHENA_DB}.iui GROUP BY 1")
iiu_ath_files['file_base'] = iiu_ath_files['file_name'].apply(lambda x: re.split(r'[/\\]', str(x))[-1])
iiu_ath_files = iiu_ath_files.groupby('file_base')['athena_rows'].sum().reset_index()

iiu_file_comp = pd.merge(iiu_loc_files, iiu_ath_files, on='file_base', how='outer').fillna(0)
iiu_file_comp['local_rows'] = iiu_file_comp['local_rows'].astype(int)
iiu_file_comp['athena_rows'] = iiu_file_comp['athena_rows'].astype(int)
iiu_file_comp['delta'] = iiu_file_comp['local_rows'] - iiu_file_comp['athena_rows']
iiu_diffs = iiu_file_comp[iiu_file_comp['delta'] != 0].sort_values(by='delta', ascending=False)
if len(iiu_diffs) > 0:
    print(f"Files with differences ({len(iiu_diffs)}):")
    display(iiu_diffs)
else:
    print("All files have 100% exact identical row counts!")

print("\n=== IIU: PREGNANCY TEST OUTCOMES COMPARISON ===")
iiu_loc_res = run_duck('''
    SELECT 
        CASE 
            WHEN UPPER(result) LIKE '%POS%' OR UPPER(result) LIKE '%GRAV%' THEN 'POSITIVO'
            WHEN UPPER(result) LIKE '%NEG%' OR UPPER(result) LIKE '%NÃO%' OR UPPER(result) LIKE '%NAO%' THEN 'NEGATIVO'
            ELSE 'OUTROS / NÃO INFORMADO'
        END as outcome_group,
        COUNT(*) as local_count
    FROM silver.planilha_embriologia_iiu
    GROUP BY 1 ORDER BY 2 DESC
''')

iiu_ath_res = run_athena(f'''
    SELECT 
        CASE 
            WHEN UPPER(result) LIKE '%POS%' OR UPPER(result) LIKE '%GRAV%' THEN 'POSITIVO'
            WHEN UPPER(result) LIKE '%NEG%' OR UPPER(result) LIKE '%NÃO%' OR UPPER(result) LIKE '%NAO%' THEN 'NEGATIVO'
            ELSE 'OUTROS / NÃO INFORMADO'
        END as outcome_group,
        COUNT(*) as athena_count
    FROM {ATHENA_DB}.iui
    GROUP BY 1 ORDER BY 2 DESC
''')

display(pd.merge(iiu_loc_res, iiu_ath_res, on='outcome_group', how='outer').fillna(0))


## 🔬 Part 7: Dedicated Clinical Streams (DOADORAS & FP SÊMEN)
Comparing `silver.planilha_embriologia_doadoras` and `silver.planilha_embriologia_fp_semen` with Athena (`{ATHENA_DB}.doadoras`, `{ATHENA_DB}.fp_semen`).


In [ ]:
print("=== DOADORAS: METRIC SUMMARY (LOCAL vs ATHENA) ===")
doad_loc = run_duck('''
    SELECT 
        'Local Silver DuckDB' as source,
        COUNT(*) as total_donor_cycles,
        COUNT(DISTINCT pin) as unique_donors,
        SUM(TRY_CAST(opu AS BIGINT)) as total_oocytes_captured,
        SUM(TRY_CAST(mii_total AS BIGINT)) as total_mature_mii
    FROM silver.planilha_embriologia_doadoras
''')

doad_ath = run_athena(f'''
    SELECT 
        'Athena {ATHENA_ENV.capitalize()}' as source,
        COUNT(*) as total_donor_cycles,
        COUNT(DISTINCT pin) as unique_donors,
        SUM(TRY_CAST(opu AS BIGINT)) as total_oocytes_captured,
        SUM(TRY_CAST(total_de_mii AS BIGINT)) as total_mature_mii
    FROM {ATHENA_DB}.doadoras
''')
display(pd.concat([doad_loc, doad_ath], ignore_index=True))

print("\n=== FP SÊMEN: SPERM CRYOPRESERVATION SUMMARY (LOCAL vs ATHENA) ===")
semen_loc = run_duck('''
    SELECT 
        'Local Silver DuckDB' as source,
        COUNT(*) as total_procedures,
        COUNT(DISTINCT pin) as unique_patients
    FROM silver.planilha_embriologia_fp_semen
''')

semen_ath = run_athena(f'''
    SELECT 
        'Athena {ATHENA_ENV.capitalize()}' as source,
        COUNT(*) as total_procedures,
        COUNT(DISTINCT pin) as unique_patients
    FROM {ATHENA_DB}.fp_semen
''')
display(pd.concat([semen_loc, semen_ath], ignore_index=True))

print("\n=== FP SÊMEN: CANCER PATHOLOGY DISTRIBUTION (LOCAL vs ATHENA) ===")
semen_loc_cancer = run_duck('''
    SELECT 
        COALESCE(tipo_cancer, 'NÃO INFORMADO / NÃO ONCOLÓGICO') as cancer_type,
        COUNT(*) as local_count
    FROM silver.planilha_embriologia_fp_semen
    GROUP BY 1 ORDER BY 2 DESC LIMIT 6
''')

semen_ath_cancer = run_athena(f'''
    SELECT 
        COALESCE(tipo_cancer, 'NÃO INFORMADO / NÃO ONCOLÓGICO') as cancer_type,
        COUNT(*) as athena_count
    FROM {ATHENA_DB}.fp_semen
    GROUP BY 1 ORDER BY 2 DESC LIMIT 6
''')
display(pd.merge(semen_loc_cancer, semen_ath_cancer, on='cancer_type', how='outer').fillna(0))


## 👥 Part 8: Patient Overlap & Identity Resolution (Strategy L)
Analysis of patient PIN overlap between Local DuckDB and Athena:


In [ ]:
import re
patient_overlap_rows = []

for s_label, loc_t, ath_t in STREAM_CONFIGS:
    loc_pins = set(re.sub(r'\.0$', '', str(r[0]).strip().lower()) for r in run_duck(f"SELECT DISTINCT pin FROM silver.{loc_t} WHERE pin IS NOT NULL").values if r[0] and str(r[0]).strip() not in ['None', 'nan', ''])
    ath_pins = set(re.sub(r'\.0$', '', str(r[0]).strip().lower()) for r in run_athena(f"SELECT DISTINCT pin FROM {ATHENA_DB}.{ath_t} WHERE pin IS NOT NULL").values if r[0] and str(r[0]).strip() not in ['None', 'nan', ''])
    
    overlap = len(loc_pins.intersection(ath_pins))
    loc_only = len(loc_pins - ath_pins)
    ath_only = len(ath_pins - loc_pins)
    union_pins = loc_pins.union(ath_pins)
    jaccard = (overlap / len(union_pins) * 100) if union_pins else 0.0
    ath_in_loc = (overlap / len(ath_pins) * 100) if ath_pins else 0.0
    loc_in_ath = (overlap / len(loc_pins) * 100) if loc_pins else 0.0
    
    patient_overlap_rows.append({
        'Procedure Stream': s_label,
        'Local Distinct PINs': f"{len(loc_pins):,}",
        f'Athena {ATHENA_ENV.capitalize()} PINs': f"{len(ath_pins):,}",
        'Common Overlapping PINs': f"{overlap:,}",
        'Local Only PINs': f"{loc_only:,}",
        f'Athena {ATHENA_ENV.capitalize()} Only PINs': f"{ath_only:,}",
        f'Athena in Local (%)': f"{ath_in_loc:.2f}%",
        f'Local in Athena (%)': f"{loc_in_ath:.2f}%",
        'Jaccard Overlap (%)': f"{jaccard:.2f}%"
    })

display(pd.DataFrame(patient_overlap_rows))


### 8.1 Strategy L (Clinisys Prontuário Matching) Match Rate per Unit-Year

Strategy L resolves patient identities between Planilha Embriologia procedure records and Clinisys master records using:
1. Normalized full name and date of birth matching.
2. Direct ID resolution (`PIN` / `chart_or_pin` $\rightarrow$ `prontuario`).
3. CPF verification.
4. Pre-normalized spelling and phonetic tolerance (Levenshtein $\le 1$).
5. Couple / spousal relationship fallbacks.

The tables below evaluate Strategy L match rates between **Local Silver (DuckDB)** and **AWS Athena Production** per **Clinic Unit** and **Procedure Year** across all **43,802 procedure cycles**.

In [ ]:
# Consolidated Strategy L Match Rates per Unit-Year (Local DuckDB vs AWS Athena Prod)
unidade_sql = '''
CASE
    WHEN LOWER(file_name) LIKE '%ibira%' OR LOWER(file_name) LIKE '%casos%ibi%' OR LOWER(file_name) LIKE '%ibi.xlsx%' OR LOWER(file_name) LIKE '%ibi 2%' THEN 'ibirapuera'
    WHEN LOWER(file_name) LIKE '%bh%' THEN 'belo_horizonte'
    WHEN LOWER(file_name) LIKE '%bsb%' THEN 'brasilia'
    WHEN LOWER(file_name) LIKE '%sj%' THEN 'santa_joana'
    WHEN LOWER(file_name) LIKE '%ssa%' THEN 'salvador'
    WHEN LOWER(file_name) LIKE '%vm%' THEN 'vila_mariana'
    ELSE 'other'
END
'''

loc_dfs = []
for s_label, loc_t, _ in STREAM_CONFIGS:
    df_s = run_duck(f'''
        SELECT 
            '{s_label}' as stream,
            {unidade_sql} as unidade,
            COALESCE(REGEXP_EXTRACT(file_name, '202[1-6]'), 'Unknown') as year,
            pin,
            CASE WHEN prontuario IS NOT NULL AND prontuario != -1 AND CAST(prontuario AS VARCHAR) != '' THEN 1 ELSE 0 END as is_matched
        FROM silver.{loc_t}
    ''')
    loc_dfs.append(df_s)
all_loc = pd.concat(loc_dfs, ignore_index=True)

ath_dfs = []
for s_label, _, ath_t in STREAM_CONFIGS:
    df_s = run_athena(f'''
        SELECT 
            '{s_label}' as stream,
            unidade,
            CAST(year AS VARCHAR) as year,
            pin,
            CASE WHEN prontuario IS NOT NULL AND CAST(prontuario AS VARCHAR) NOT IN ('-1', '', 'null') THEN 1 ELSE 0 END as is_matched
        FROM {ATHENA_DB}.{ath_t}
    ''')
    ath_dfs.append(df_s)
all_ath = pd.concat(ath_dfs, ignore_index=True)

# Aggregate per Unit-Year
loc_uy = all_loc.groupby(['unidade', 'year']).agg(
    total_cycles=('pin', 'count'),
    local_matched=('is_matched', 'sum'),
    distinct_patients=('pin', lambda s: s.astype(str).str.lower().str.strip().nunique()),
    matched_patients=('pin', lambda s: s[all_loc.loc[s.index, 'is_matched'] == 1].astype(str).str.lower().str.strip().nunique())
).reset_index()
loc_uy['local_match_rate_%'] = (loc_uy['local_matched'] / loc_uy['total_cycles'] * 100).round(2)
loc_uy['patient_match_rate_%'] = (loc_uy['matched_patients'] / loc_uy['distinct_patients'] * 100).round(2)

ath_uy = all_ath.groupby(['unidade', 'year']).agg(
    athena_cycles=('pin', 'count'),
    athena_matched=('is_matched', 'sum')
).reset_index()
ath_uy['athena_match_rate_%'] = (ath_uy['athena_matched'] / ath_uy['athena_cycles'] * 100).round(2)

# Merge Local vs Athena comparison
comp_uy = pd.merge(loc_uy, ath_uy, on=['unidade', 'year'], how='outer').fillna(0)
comp_uy['delta_matched'] = comp_uy['local_matched'] - comp_uy['athena_matched']
comp_uy['delta_rate_%'] = (comp_uy['local_match_rate_%'] - comp_uy['athena_match_rate_%']).round(2)

print(f'=== STRATEGY L MATCHING RATE PER UNIT-YEAR (LOCAL SILVER vs ATHENA {ATHENA_ENV.upper()}) ===')
display(comp_uy[[
    'unidade', 'year', 'total_cycles', 'distinct_patients', 
    'local_matched', 'local_match_rate_%', 
    'athena_matched', 'athena_match_rate_%', 
    'delta_matched', 'delta_rate_%'
]])

# Pivot Matrix: Unidade vs Year Match Rate %
pivot_tot = all_loc.pivot_table(index='unidade', columns='year', values='pin', aggfunc='count', margins=True, margins_name='All Units')
pivot_mat = all_loc.pivot_table(index='unidade', columns='year', values='is_matched', aggfunc='sum', margins=True, margins_name='All Units')
pivot_pct = (pivot_mat / pivot_tot * 100).round(2)

print('\n=== STRATEGY L MATCH RATE (%) PIVOT MATRIX (BY UNIT AND YEAR) ===')
display(pivot_pct.apply(lambda col: col.map(lambda v: f'{v:.2f}%' if pd.notnull(v) else '-')))


### 8.2 Strategy L Match Rate by Clinical Procedure Stream

Breakdown of Strategy L matching performance across the 8 procedure streams (`FET`, `FRESH`, `RECEP`, `FOT`, `DOADORAS`, `FP_SEMEN`, `FP_OVULOS`, `IIU`), illustrating how match rates perform across specialized workflows.

In [ ]:
# Strategy L Match Rate by Procedure Stream
stream_summary = all_loc.groupby('stream').agg(
    total_procedures=('pin', 'count'),
    matched_prontuarios=('is_matched', 'sum'),
    distinct_pins=('pin', lambda s: s.astype(str).str.lower().str.strip().nunique()),
    matched_pins=('pin', lambda s: s[all_loc.loc[s.index, 'is_matched'] == 1].astype(str).str.lower().str.strip().nunique())
).reset_index()

stream_summary['cycle_match_rate_%'] = (stream_summary['matched_prontuarios'] / stream_summary['total_procedures'] * 100).round(2)
stream_summary['patient_match_rate_%'] = (stream_summary['matched_pins'] / stream_summary['distinct_pins'] * 100).round(2)

# Sort by procedure volume
stream_summary = stream_summary.sort_values(by='total_procedures', ascending=False).reset_index(drop=True)

# Add Athena match counts for comparison
ath_stream = all_ath.groupby('stream').agg(
    athena_matched=('is_matched', 'sum')
).reset_index()
stream_summary = pd.merge(stream_summary, ath_stream, on='stream', how='left')
stream_summary['athena_match_rate_%'] = (stream_summary['athena_matched'] / stream_summary['total_procedures'] * 100).round(2)
stream_summary['delta_matched'] = stream_summary['matched_prontuarios'] - stream_summary['athena_matched']

print('=== STRATEGY L MATCH RATE BY PROCEDURE STREAM ===')
display(stream_summary[[
    'stream', 'total_procedures', 'distinct_pins',
    'matched_prontuarios', 'cycle_match_rate_%',
    'athena_matched', 'athena_match_rate_%', 'delta_matched',
    'patient_match_rate_%'
]])

# Stream by Unit Match Rate Matrix
stream_unit_tot = all_loc.pivot_table(index='stream', columns='unidade', values='pin', aggfunc='count', margins=True, margins_name='All Units')
stream_unit_mat = all_loc.pivot_table(index='stream', columns='unidade', values='is_matched', aggfunc='sum', margins=True, margins_name='All Units')
stream_unit_pct = (stream_unit_mat / stream_unit_tot * 100).round(2)

print('\n=== STRATEGY L MATCH RATE (%) BY STREAM AND UNIT ===')
display(stream_unit_pct.apply(lambda col: col.map(lambda v: f'{v:.2f}%' if pd.notnull(v) else '-')))
